# 02 索引

## 八股问答

### 1. 什么是索引？索引的作用和类型？

索引是数据库维护的一种额外数据结构，让查询能按序定位而不是全表扫描。

作用：

- 加速查询：按索引树走，读取行数从全表降到对数级。
- 加速排序和分组：索引本身有序，`ORDER BY`、`GROUP BY` 可免排序。
- 唯一性约束：唯一索引直接保证字段不重复。

常见索引分类：

| 索引类型 | 含义 | 特点 |
|---|---|---|
| 主键索引 | 以主键列建立的索引，InnoDB 中叶子节点直接存整行数据。 | 唯一且非空，一张表最多一个，按主键定位最快。 |
| 唯一索引 | 用于保证索引列值不重复的索引。 | 允许 NULL、可建多个，既做唯一性约束也加速查询。 |
| 常规索引 | 不附加唯一或主键约束、只为加速查询建立的索引。 | 字段值可重复、一张表可建多个，是最常用的查询加速索引。 |
| 联合索引 | 由多个列共同组成的索引。 | 查询需遵循最左前缀原则，能同时加速多个条件的组合查询。 |
| 全文索引 | 针对文本内容建立的关键词索引。 | 配合全文检索函数使用，中文通常需要配置分词。 |

在 InnoDB 中，按索引的存储形式还可以分为：

| 索引类型 | 含义 | 特点 |
|---|---|---|
| 聚簇索引 | 底层是 B+ 树，叶子节点直接存整行数据。 | 表数据按主键有序存放，通常每张表只有一个，按主键查询无需回表。 |
| 二级索引 | 底层也是 B+ 树，叶子节点只存索引列和主键值，不存整行。 | 可以建多个，查询非索引列时通常需要回表取整行。 |

![聚簇索引与二级索引结构示意图](assets/index-clustered-secondary.png)

这就是为什么叫“二级”：

根据 name 查询：

```text
name 索引 → 找到 id → 再用 id 查聚簇索引 → 找到完整行（这个过程叫回表查询）
```

### 2. B+ 树的结构？

B+ 树是平衡多路搜索树，两层结构：

- 非叶子节点：只存索引键和指向子节点的指针，不存数据，所以一个节点能装大量
  键，树更矮。
- 叶子节点：存完整的索引键 + 数据指针（二级索引存主键值），叶子之间用链表
  按序连接。

为什么矮：InnoDB 一个页 16KB，三层 B+ 树就能支撑千万级数据，查询约 3 次磁盘
IO。为什么叶子用链表：范围查询只需从起点顺序扫，不需要反复走树。

![B+ 树结构示意图](assets/b-plus-tree-structure.png)

![B+ 树高度计算示例](assets/b-plus-tree-height-calculation.png)

### 3. Hash 索引的特点？

- 只能进行等值比较，如 `=`、`IN`。
- 不支持范围查询，如 `BETWEEN`、`>`、`<` 等。
- 无法利用索引完成排序操作。
- 查询高效，通常只需要一次检索，效率通常高于 B+ 树索引。

### 4. 为什么索引用 B+ 树而不是 B 树、哈希、二叉树？

- 哈希：等值查询 O(1)，但无法范围查询、无法排序。
- 二叉树：深度随数据量增长，磁盘 IO 次数多；极端情况退化成链表。
- B 树：所有节点都存数据，相同容量下高度更高；范围查询需要中序遍历。
- B+ 树：非叶子只存键、叶子有序链表连接，高度低、范围查询快，最适合磁盘。

### 5. 聚簇索引、二级索引、回表是什么？

- 聚簇索引：主键索引，叶子节点直接存整行数据，表数据按主键有序存放。
- 二级索引（普通索引）：叶子节点存索引列 + 主键值，而不是整行。
- 回表：二级索引查到主键后，再回聚簇索引取整行，多一次索引查找。

回表的劣势是多次 IO；优势是主键可以复用在所有二级索引上，且数据不重复存储。
`EXPLAIN` 中 `Using index` 表示不用回表，`Using index condition` 表示先按索引
条件过滤再回表。

#### 二级索引叶子节点一定包含主键吗？

在 InnoDB 中，二级索引的叶子节点一定带有用于回聚簇索引的键。表有显式主键时，
这个键就是主键列；如果是联合索引，叶子节点保存的是“联合索引字段 + 主键字段”。

例如 `idx_user_created(user_id, created_at)` 的叶子节点保存：

```text
user_id, created_at, id
```

所以：

- 查询只需要 `user_id`、`created_at`、`id` 时，可以走覆盖索引，不需要回表；
- 查询还需要其他列时，先用二级索引拿到 `id`，再回聚簇索引取完整行。

如果没有主键，InnoDB 可能选用第一个 `NOT NULL UNIQUE` 索引作为聚簇索引；
如果连合适的唯一键都没有，则使用隐藏行 ID。因此更准确的说法是：二级索引叶子节点
保存的是“聚簇索引键”，通常就是主键。

### 6. 覆盖索引是什么？

查询需要的所有列都在同一个二级索引里，不需要回表，`Extra` 显示 `Using index`。
例如 `SELECT user_id, created_at FROM orders WHERE user_id = 100` 能被
`idx_user_created(user_id, created_at)` 覆盖。

![覆盖索引示例](assets/covering-index-example.png)

图中三个查询：

1. `SELECT * WHERE id = 2`：走聚簇索引，直接返回整行。
2. `SELECT id, name WHERE name = 'Arm'`：`id` 和 `name` 都在辅助索引叶子节点中，
   不需要回表，`Extra` 显示 `Using index`，这就是覆盖索引。
3. `SELECT id, name, gender WHERE name = 'Arm'`：`gender` 不在辅助索引中，
   查到 `id = 2` 后需要回聚簇索引取 `gender`，这不是覆盖索引。

### 6.1 前缀索引是什么？

对字符串或长文本字段只索引前 N 个字符，减少索引占用的存储和内存。

例如：

```sql
-- 对 email 只索引前 5 个字符
CREATE INDEX idx_email_prefix ON users(email(5));
```

![前缀索引示例](assets/prefix-index-example.png)

图中辅助索引为 `email(5)`，只把 email 前 5 个字符和 id 放进叶子节点。
查询 `WHERE email = 'lvbu666@163.com'` 先通过前缀索引命中并得到 id，
再回聚簇索引取整行并校验完整 email。前缀索引通常不是覆盖索引，因为不保存完整字段值。

### 7. 最左前缀原则？

联合索引 `(a, b, c)` 相当于建了 `a`、`(a,b)`、`(a,b,c)` 三个索引。查询条件只有
包含最左列 `a` 时才能用上；`b` 和 `c` 单独查询不会走这个索引。`a` 等值后 `b`
范围查询，`c` 也无法继续用于过滤和排序。

简单例子，假设联合索引为 `idx(a, b, c)`：

| WHERE 条件 | 索引使用情况 |
| --- | --- |
| `WHERE a = 1` | 能使用索引 |
| `WHERE a = 1 AND b = 2` | 能使用索引 |
| `WHERE b = 2` | 不能使用索引（缺少最左列 `a`） |
| `WHERE a = 1 AND c = 3` | 只能使用 `a`，`c` 用不上 |
| `WHERE a = 1 AND b > 2 AND c = 3` | `a`、`b` 能使用，`c` 不能继续用于过滤和排序 |

注意：把条件写成 `a = 1 AND c = 3 AND b > 2` 也**没有用**，SQL 书写顺序不影响索引使用。
要让 `c` 也参与索引，要么把 `b` 改成等值条件，要么把索引改成 `(a, c, b)`。
不满足最左前缀时，优化器也可能选择索引，但只能走 `type=index` 的全索引扫描（或覆盖扫描），而不是高效的 `ref`/`range` 查找。
所以设计联合索引时，尽量把等值条件列放前面，范围检索列放后面。

### 8. 哪些情况索引会失效？

不要在索引上进行运算操作，索引将失效。

例如：

```sql
-- 对索引列 name 使用 SUBSTR，索引失效
SELECT * FROM users WHERE SUBSTR(name, 1, 3) = 'abc';

-- 字符串不加引号，隐式类型转换导致索引失效
SELECT * FROM users WHERE phone = 13812345678;

-- 尾部模糊匹配（固定前缀），索引不失效
SELECT * FROM users WHERE name LIKE 'abc%';

-- 头部模糊匹配（前导通配符），索引失效
SELECT * FROM users WHERE name LIKE '%abc';

-- OR 连接非索引列，索引失效
SELECT * FROM users WHERE name = 'abc' OR status = 1;
```

只有 OR 两侧的列都有索引，才可能继续利用索引；一侧无索引时会退化为全表扫描。

另外，如果 MySQL 评估使用索引比全表扫描更慢，它也不会使用索引。

### 9. 什么时候建索引？索引如何设计？

适合建索引：

- 高频查询的过滤列、排序列、join 关联列。
  - 例如：`WHERE user_id = 123`，给 `user_id` 建索引。
- 区分度高（选择性接近 1）的列，如 order_no、email。
  - 例如：`WHERE email = 'a@example.com'`，给 `email` 建唯一索引。
- 覆盖高频查询的列组合，用联合索引避免回表。
  - 例如：`SELECT user_id, created_at ...`，建 `(user_id, created_at)`。

不适合建索引：

- 几乎不查询的列。
  - 例如：`WHERE nickname = 'xx'` 很少执行，不建议单独建索引。
- 区分度低的列，如性别，单独建索引收益很低。
  - 例如：`WHERE gender = '男'` 只区分两种值，收益很低。
- 写入频繁且读少的表：每次写入都要维护索引树。
  - 例如：只写不查的日志表，尽量少建索引。
- 大字段（长文本）直接建索引，只能考虑前缀索引。
  - 例如：`content TEXT` 不要直接建索引，可用 `content(50)`。

设计建议：

1. 主键用自增或有序值，避免页分裂。
   - 例如：`id BIGINT UNSIGNED AUTO_INCREMENT PRIMARY KEY`。
2. 联合索引按“等值列在前、范围列在后”排列。
   - 例如：`WHERE user_id = 1 AND created_at > '2026-01-01'`，建 `(user_id, created_at)`。
3. 用覆盖索引换掉 `SELECT *` 的热点查询。
   - 例如：把 `SELECT *` 改成 `SELECT user_id, created_at`，并建 `(user_id, created_at)`。
4. 控制单表索引数量，索引不是越多越好。
   - 例如：不要给每个可能查询的列都建索引，先看 `EXPLAIN` 实际使用情况。

### 10. 如何分页比较高效？

普通深分页 `LIMIT 50000, 20` 要扫描并丢弃前 5 万行，慢在“读了很多行但只返回
20 行”。优化方式：

- 延迟关联：先只查主键（覆盖索引），再 join 回表取整行。
- 游标分页：用上一页最后一条的排序字段做 `WHERE id > ?`，稳定且 O(1) 量级。

详细实操见 [09-性能优化与EXPLAIN](09-性能优化与EXPLAIN.md)。


## 面试话术

“我先把主键设计成自增的，让 InnoDB 顺序写入；查询按最左前缀建联合索引，并优先
用覆盖索引避免回表。上线前用 EXPLAIN 确认 type 不是 ALL、key 真的用上，再配合
慢查询日志抓漏网之鱼。索引不是越多越好，写多读少的表要控制索引数量，因为每次
写入都要维护索引树。”


## 自查清单

- [ ] 能画出 B+ 树结构和聚簇索引/二级索引的关系
- [ ] 能解释 `Using index` 和 `Using index condition` 的区别
- [ ] 能在 `orders` 表上亲手跑出 const、ref、range、ALL 四种 type
- [ ] 能说出至少四种索引失效场景并在终端验证
- [ ] 能给出一个具体表的索引设计并说明取舍
